# Inverse distance weighting: explore the surface

A few observations are known. What should we estimate between them?

This lab lets you move the target location, change the distance power and limit the number of neighbours. Watch how those choices alter the estimate and the surrounding surface.

**You need:** Python, NumPy, Matplotlib and ipywidgets. Run the cells in order.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, Dropdown, interact

points = np.array([
    [1.0, 1.0], [3.0, 1.0], [5.0, 2.0],
    [2.0, 4.0], [4.0, 4.0], [6.0, 5.0],
])
values = np.array([12.0, 15.0, 22.0, 18.0, 30.0, 34.0])

## 1. See the observations

Each point has a measured value. The surface is not observed between these points; it will be estimated.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(points[:, 0], points[:, 1], c=values, s=180, edgecolor="black")
for (x, y), value in zip(points, values):
    ax.annotate(f"{value:g}", (x, y), xytext=(6, 6), textcoords="offset points")
fig.colorbar(scatter, ax=ax, label="Observed value")
ax.set(xlim=(0, 7), ylim=(0, 6), xlabel="x coordinate", ylabel="y coordinate", title="Six observed locations")
ax.set_aspect("equal")
ax.grid(alpha=0.2)
plt.show()

## 2. Calculate one estimate

For each target, IDW calculates a distance to every contributing observation, converts those distances into inverse-power weights, normalises the weights and takes a weighted average.

The notebook uses Euclidean distance in this small coordinate plane.

In [ ]:
def idw(target, power=2.0, max_neighbors=None):
    distances = np.linalg.norm(points - np.asarray(target, dtype=float), axis=1)
    exact = np.flatnonzero(distances == 0)
    if len(exact):
        return float(values[exact].mean())
    order = np.argsort(distances)
    if max_neighbors is not None:
        order = order[:max_neighbors]
    selected_distances = distances[order]
    selected_values = values[order]
    weights = (selected_distances.min() / selected_distances) ** power
    return float(np.sum(weights * selected_values) / weights.sum())

target = (3.0, 2.5)
print(f"Estimate at {target}: {idw(target):.2f}")

## 3. Change the target, power and neighbourhood

Move the target across the area. Increase the power to make the nearest observations more influential. Reduce the neighbour count to restrict the estimate to the closest points.

The coloured surface is recalculated from the same six measurements each time. It is a teaching example, not a validated rainfall or environmental model.

In [ ]:
grid_x, grid_y = np.meshgrid(np.linspace(0, 7, 45), np.linspace(0, 6, 40))

def explore(target_x=3.0, target_y=2.5, power=2.0, neighbours="All observations"):
    k = None if neighbours == "All observations" else int(neighbours)
    surface = np.empty_like(grid_x)
    for row in range(grid_x.shape[0]):
        for col in range(grid_x.shape[1]):
            surface[row, col] = idw((grid_x[row, col], grid_y[row, col]), power, k)

    estimate = idw((target_x, target_y), power, k)
    fig, ax = plt.subplots(figsize=(8, 5.5))
    filled = ax.contourf(grid_x, grid_y, surface, levels=18)
    ax.scatter(points[:, 0], points[:, 1], c=values, cmap=filled.cmap,
               vmin=values.min(), vmax=values.max(), edgecolor="black", s=110)
    ax.scatter([target_x], [target_y], marker="*", s=260, color="white",
               edgecolor="black", label=f"Estimate: {estimate:.2f}")
    ax.legend()
    fig.colorbar(filled, ax=ax, label="IDW estimate")
    ax.set(xlabel="x coordinate", ylabel="y coordinate",
           title=f"IDW surface (power={power:g}, neighbours={neighbours})")
    ax.set_aspect("equal")
    plt.show()
    print(f"Estimated value at target: {estimate:.3f}")

interact(
    explore,
    target_x=FloatSlider(value=3.0, min=0, max=7, step=0.25, description="Target x"),
    target_y=FloatSlider(value=2.5, min=0, max=6, step=0.25, description="Target y"),
    power=FloatSlider(value=2.0, min=0.5, max=5.0, step=0.25, description="Power"),
    neighbours=Dropdown(options=["All observations", "3", "4", "5"],
                        value="All observations", description="Neighbours"),
);

## 4. Questions to investigate

1. What happens to the surface as the power increases?
2. Which observations influence the target when you choose three neighbours?
3. Does the estimate ever fall outside the range of contributing values?
4. What changes when the target is placed directly on an observation?
5. Where would you avoid trusting the surface?

## 5. Important limits

This example uses a flat Euclidean coordinate plane. Real geographic work needs a suitable distance model, attention to barriers and sampling density, and validation at withheld locations. IDW does not provide a model-based prediction variance.

Continue with the [IDW lesson](../algorithms/interpolation/inverse-distance-weighting.md) and its [tested Python implementation](../implementations/python/inverse-distance-weighting/README.md).